In [ ]:
# Load dataset
import pandas as pd
import numpy as np

# --- LOADING AND CLEANING ---
file_path = "ALLFLOWMETER_HIKARI2021.csv"
df = pd.read_csv(file_path)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df_filtrado = df

In [ ]:
%%writefile ids_engine2.py
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import multiprocessing as mp
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE

# ------------------------------------------------------------
# MEMORY PROTECTIONS
# ------------------------------------------------------------
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ------------------------------------------------------------
# GENERAL SETTINGS
# ------------------------------------------------------------
BATCH_SIZE = 32
ADV_BATCH_SIZE = 64
EPOCHS = 100
EPSILONS = [0.001, 0.005, 0.01, 0.02, 0.05]
N_RUNS = 30

# ------------------------------------------------------------
# HIKARI STRICT PLAUSIBILITY MASK
# ------------------------------------------------------------
# This mask follows a deny-by-default strategy.
# All input features are blocked by default, and only features explicitly
# listed in HIKARI_ALLOWED_FEATURES can be perturbed.
#
# Plausibility criterion adopted here:
# - Label and traffic_category are response variables and are never perturbed.
# - Identifiers and topology/session metadata are not perturbed.
# - TCP flags, TCP window sizes, header sizes, payload primitive statistics,
#   packet counters, bulk and subflow features are blocked in this strict mask.
# - The attack is restricted to timing/rate features that can be plausibly
#   affected by delaying, pacing, or spacing packets.
#
# Supported dataset layouts:
# 1. Full HIKARI dataframe with 86 columns:
#    5 identifier columns + 79 numerical traffic features + traffic_category + Label.
# 2. Legacy/full dataframe with 88 columns:
#    7 metadata columns + 79 numerical traffic features + traffic_category + Label.
# 3. Filtered dataframe:
#    79 numerical traffic features + traffic_category + Label.
#
# Mask semantics:
# - 1.0 means that the feature may be perturbed.
# - 0.0 means that the feature is blocked and must remain unchanged.

HIKARI_EXPECTED_FEATURES = 79
HIKARI_RESPONSE_COLUMNS = ["traffic_category", "Label"]

# Canonical order of the 79 numerical HIKARI traffic features, excluding:
# uid, originh, originp, responh, responp, traffic_category, and Label.
HIKARI_NUMERICAL_FEATURES = [
    "flow_duration",
    "fwd_pkts_tot",
    "bwd_pkts_tot",
    "fwd_data_pkts_tot",
    "bwd_data_pkts_tot",
    "fwd_pkts_per_sec",
    "bwd_pkts_per_sec",
    "flow_pkts_per_sec",
    "down_up_ratio",
    "fwd_header_size_tot",
    "fwd_header_size_min",
    "fwd_header_size_max",
    "bwd_header_size_tot",
    "bwd_header_size_min",
    "bwd_header_size_max",
    "flow_FIN_flag_count",
    "flow_SYN_flag_count",
    "flow_RST_flag_count",
    "fwd_PSH_flag_count",
    "bwd_PSH_flag_count",
    "flow_ACK_flag_count",
    "fwd_URG_flag_count",
    "bwd_URG_flag_count",
    "flow_CWR_flag_count",
    "flow_ECE_flag_count",
    "fwd_pkts_payload.min",
    "fwd_pkts_payload.max",
    "fwd_pkts_payload.tot",
    "fwd_pkts_payload.avg",
    "fwd_pkts_payload.std",
    "bwd_pkts_payload.min",
    "bwd_pkts_payload.max",
    "bwd_pkts_payload.tot",
    "bwd_pkts_payload.avg",
    "bwd_pkts_payload.std",
    "flow_pkts_payload.min",
    "flow_pkts_payload.max",
    "flow_pkts_payload.tot",
    "flow_pkts_payload.avg",
    "flow_pkts_payload.std",
    "fwd_iat.min",
    "fwd_iat.max",
    "fwd_iat.tot",
    "fwd_iat.avg",
    "fwd_iat.std",
    "bwd_iat.min",
    "bwd_iat.max",
    "bwd_iat.tot",
    "bwd_iat.avg",
    "bwd_iat.std",
    "flow_iat.min",
    "flow_iat.max",
    "flow_iat.tot",
    "flow_iat.avg",
    "flow_iat.std",
    "payload_bytes_per_second",
    "fwd_subflow_pkts",
    "bwd_subflow_pkts",
    "fwd_subflow_bytes",
    "bwd_subflow_bytes",
    "fwd_bulk_bytes",
    "bwd_bulk_bytes",
    "fwd_bulk_packets",
    "bwd_bulk_packets",
    "fwd_bulk_rate",
    "bwd_bulk_rate",
    "active.min",
    "active.max",
    "active.tot",
    "active.avg",
    "active.std",
    "idle.min",
    "idle.max",
    "idle.tot",
    "idle.avg",
    "idle.std",
    "fwd_init_window_size",
    "bwd_init_window_size",
    "fwd_last_windows_size",
]

HIKARI_ALLOWED_FEATURES = [
    # Flow duration and packet-rate features.
    "flow_duration",
    "fwd_pkts_per_sec",
    "bwd_pkts_per_sec",
    "flow_pkts_per_sec",

    # Forward inter-arrival time features.
    "fwd_iat.min",
    "fwd_iat.max",
    "fwd_iat.tot",
    "fwd_iat.avg",
    "fwd_iat.std",

    # Backward inter-arrival time features.
    "bwd_iat.min",
    "bwd_iat.max",
    "bwd_iat.tot",
    "bwd_iat.avg",
    "bwd_iat.std",

    # Bidirectional flow inter-arrival time features.
    "flow_iat.min",
    "flow_iat.max",
    "flow_iat.tot",
    "flow_iat.avg",
    "flow_iat.std",

    # Payload rate. This is kept because it changes with timing/pacing.
    "payload_bytes_per_second",

    # Active-time features.
    "active.min",
    "active.max",
    "active.tot",
    "active.avg",
    "active.std",

    # Idle-time features.
    "idle.min",
    "idle.max",
    "idle.tot",
    "idle.avg",
    "idle.std",
]


def select_hikari_features_and_target(df):
    """
    Selects the 79 numerical HIKARI traffic features and the binary target.

    Supported inputs:
    1. Full HIKARI dataframe:
       5 identifier columns + 79 features + traffic_category + Label.
    2. Legacy/full dataframe:
       7 metadata columns + 79 features + traffic_category + Label.
    3. Filtered dataframe:
       79 features + traffic_category + Label.

    The columns traffic_category and Label are response variables, not input
    features. Therefore, they are never included in X or in the plausibility mask.
    """
    if "Label" in df.columns:
        y = df["Label"].astype("int32").copy()
    else:
        y = df.iloc[:, -1].astype("int32").copy()

    candidate_feature_groups = []

    # Best case: named columns with the canonical HIKARI feature names.
    if set(HIKARI_NUMERICAL_FEATURES).issubset(set(df.columns)):
        candidate_feature_groups.append(HIKARI_NUMERICAL_FEATURES)

    # Full HIKARI layout described in the LaTeX document:
    # 5 identifiers + 79 numerical features + 2 response columns = 86 columns.
    if df.shape[1] >= 86:
        candidate_feature_groups.append(list(df.columns[5:-2]))

    # Legacy layout used in some notebooks:
    # 7 metadata columns + 79 numerical features + 2 response columns = 88 columns.
    if df.shape[1] >= 88:
        candidate_feature_groups.append(list(df.columns[7:-2]))

    # Filtered layout:
    # 79 numerical features + traffic_category + Label = 81 columns.
    if df.shape[1] >= 81:
        candidate_feature_groups.append(list(df.columns[:-2]))

    # Explicit response-column removal. This works when the dataframe contains
    # only the 79 features plus response columns.
    response_cols_present = [
        col for col in HIKARI_RESPONSE_COLUMNS
        if col in df.columns
    ]
    if response_cols_present:
        candidate_feature_groups.append(
            [col for col in df.columns if col not in response_cols_present]
        )

    for feature_cols in candidate_feature_groups:
        feature_cols = list(feature_cols)

        if len(feature_cols) != HIKARI_EXPECTED_FEATURES:
            continue

        if any(col in HIKARI_RESPONSE_COLUMNS for col in feature_cols):
            continue

        if set(HIKARI_ALLOWED_FEATURES).issubset(set(feature_cols)):
            X = df.loc[:, feature_cols].copy()
            return X, y, feature_cols

    raise ValueError(
        "Could not select the 79 HIKARI input features. Expected either the full "
        "dataset format (5 identifiers + 79 features + traffic_category + Label), "
        "the legacy full format (7 metadata columns + 79 features + traffic_category "
        "+ Label), or the filtered format (79 features + traffic_category + Label)."
    )


def build_hikari_plausibility_mask(feature_cols):
    """
    Builds a strict HIKARI plausibility mask over the 79 numerical input features.

    Mask semantics:
    - 1.0 means the feature may be perturbed.
    - 0.0 means the feature is blocked and must remain unchanged.

    This function uses a deny-by-default strategy:
    - all features are blocked by default;
    - only HIKARI_ALLOWED_FEATURES are marked as perturbable.

    In this strict temporal/rate mask, the attack is allowed to modify only
    features related to duration, packet rates, inter-arrival times, active/idle
    periods, and payload rate. This avoids direct independent modification of:
    - response variables such as traffic_category and Label;
    - identifiers and topology/session metadata;
    - TCP flags, window sizes, and header-size fields;
    - packet counters, primitive payload statistics, bulk, and subflow features.
    """
    feature_cols = list(feature_cols)

    if len(feature_cols) != HIKARI_EXPECTED_FEATURES:
        raise ValueError(
            f"Expected {HIKARI_EXPECTED_FEATURES} input features, "
            f"but received {len(feature_cols)}."
        )

    leaked_targets = [
        col for col in HIKARI_RESPONSE_COLUMNS
        if col in feature_cols
    ]
    if leaked_targets:
        raise ValueError(
            "Response columns found among input features: "
            + ", ".join(leaked_targets)
            + ". traffic_category and Label must not be part of the mask."
        )

    mask_series = pd.Series(0.0, index=feature_cols, dtype="float32")

    missing_features = [
        feature for feature in HIKARI_ALLOWED_FEATURES
        if feature not in mask_series.index
    ]

    if missing_features:
        raise ValueError(
            "The following expected HIKARI allowed features were not found: "
            + ", ".join(missing_features)
        )

    mask_series.loc[HIKARI_ALLOWED_FEATURES] = 1.0

    return mask_series.values.astype("float32"), mask_series


def print_plausibility_mask_report(mask_series):
    """
    Prints a compact report for the strict plausibility mask.
    """
    blocked = mask_series[mask_series == 0.0]
    allowed = mask_series[mask_series == 1.0]

    print("\n" + "-" * 60)
    print("[HIKARI Strict Temporal/Rate Plausibility Mask]")
    print(f"Total input features: {len(mask_series)}")
    print(f"Perturbable features: {len(allowed)}")
    print(f"Blocked features: {len(blocked)}")

    print("\nPerturbable feature names:")
    for idx, name in enumerate(allowed.index):
        print(f"  {idx + 1:02d}. {name}")

    print("\nBlocked feature names:")
    for idx, name in enumerate(blocked.index):
        print(f"  {idx + 1:02d}. {name}")

    print("-" * 60)

def configure_gpu():
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            pass

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def build_mtl_model_M1(input_shape):
    inputs = layers.Input(shape=input_shape)

    def eca_block(input_tensor):
        channels = input_tensor.shape[-1]
        squeeze = layers.GlobalAveragePooling2D()(input_tensor)
        squeeze = layers.Reshape((1, 1, channels))(squeeze)
        k_size = max(3, int(abs((np.log2(channels) + 1) / 2 + 0.5)))
        squeeze = layers.Conv2D(1, kernel_size=(1, k_size), padding="same", activation="sigmoid", use_bias=False)(squeeze)
        return layers.Multiply()([input_tensor, squeeze])

    def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.3):
        x = layers.LayerNormalization(epsilon=1e-6)(inputs)
        x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
        x = layers.Dropout(dropout)(x)
        res = x + inputs
        x = layers.LayerNormalization(epsilon=1e-6)(res)
        x = layers.Dense(ff_dim, activation="relu")(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Dense(inputs.shape[-1])(x)
        return x + res

   # Convolutional Block 1
    x = layers.Conv2D(32, (3, 3), padding="same")(inputs)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Convolutional Block 2
    x = layers.Conv2D(64, (3, 3), padding="same")(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Convolutional Block 3
    x = layers.Conv2D(128, (3, 3), padding="same")(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    x = layers.Reshape((-1, x.shape[-1]))(x)
    x = transformer_encoder(x, head_size=128, num_heads=4, ff_dim=256, dropout=0.3)
    x = layers.Flatten()(x)

    # L2 regularization (1e-2)
    x = layers.Dense(128, activation="relu", kernel_regularizer=l2(1e-2))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=l2(1e-2))(x)

    # [FIX] Separation of logits and final activation
    logits = layers.Dense(1, activation=None, name="logits")(x)
    output = layers.Activation("sigmoid", dtype="float32", name="binary_output")(logits)

    model = Model(inputs=inputs, outputs=output)

    # Learning rate reduction to 1e-4
    model.compile(optimizer=Adam(learning_rate=0.0001), loss=BinaryCrossentropy(), metrics=["accuracy", "Precision", "Recall", "AUC"])
    return model

# ------------------------------------------------------------
# METRICS AND THRESHOLD
# ------------------------------------------------------------
def get_best_threshold(y_true, y_pred_proba):
    best_t = 0.5
    best_f1 = 0.0
    y_true_flat = np.asarray(y_true).reshape(-1)
    y_pred_proba_flat = np.asarray(y_pred_proba).reshape(-1)

    for t in np.arange(0.01, 1.0, 0.01):
        y_pred = (y_pred_proba_flat > t).astype("int32")
        current_f1 = f1_score(y_true_flat, y_pred, zero_division=0)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_t = t

    return best_t

def get_metrics(y_true, y_pred_proba, threshold):
    y_true, y_pred_proba = np.asarray(y_true).reshape(-1), np.asarray(y_pred_proba).reshape(-1)
    y_pred = (y_pred_proba > threshold).astype("int32")
    return {
        "Acc": accuracy_score(y_true, y_pred), "Prec": precision_score(y_true, y_pred, zero_division=0),
        "Rec": recall_score(y_true, y_pred, zero_division=0), "F1": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_pred_proba)
    }

def calculate_ASR_I(y_true, y_clean_proba, y_adv_proba, threshold):
    y_clean_pred = (np.asarray(y_clean_proba).reshape(-1) > threshold).astype("int32")
    y_adv_pred = (np.asarray(y_adv_proba).reshape(-1) > threshold).astype("int32")
    originally_malicious = (np.asarray(y_true).reshape(-1) == 1) & (y_clean_pred == 1)
    successfully_evaded = originally_malicious & (y_adv_pred == 0)
    denominator = np.sum(originally_malicious)
    return np.sum(successfully_evaded) / denominator if denominator > 0 else 0.0

def reshape_to_2d(data, size):
    pad_size = size ** 2 - data.shape[1]
    padded = np.pad(data, pad_width=((0, 0), (0, pad_size)), mode="constant")
    return padded.reshape(-1, size, size, 1).astype("float32")

def prepare_splits(X, y, seed):
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=seed, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp)

    smote = SMOTE(random_state=seed)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train_res).astype("float32")
    X_val_scaled = scaler.transform(X_val).astype("float32")
    X_test_scaled = scaler.transform(X_test).astype("float32")

    size = int(np.ceil(np.sqrt(X_train_scaled.shape[1])))
    return reshape_to_2d(X_train_scaled, size), reshape_to_2d(X_val_scaled, size), reshape_to_2d(X_test_scaled, size), np.asarray(y_train_res).astype("float32"), np.asarray(y_val).astype("float32"), np.asarray(y_test).astype("float32"), size, scaler

def make_tf_dataset(X_data, y_data, batch_size, shuffle=False, seed=42):
    X_data, y_data = X_data.astype("float32"), np.asarray(y_data).astype("float32").reshape(-1, 1)
    with tf.device('/CPU:0'):
        dataset = tf.data.Dataset.from_tensor_slices((X_data, y_data))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=min(len(X_data), 10000), seed=seed, reshuffle_each_iteration=True)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# ------------------------------------------------------------
# WHITE-BOX ADVERSARIAL ATTACKS
# ------------------------------------------------------------
def fgsm_attack_batched(logit_model, x, y, epsilon, feature_mask_2d, batch_size=256):
    """
    Generates masked FGSM adversarial examples.

    The feature mask is applied directly to the input gradient. This ensures that
    blocked HIKARI features remain unchanged during adversarial generation.
    """
    x_adv_list = []
    y_array = np.asarray(y).reshape(-1).astype("float32")
    mask_tensor = tf.convert_to_tensor(feature_mask_2d, dtype=tf.float32)
    bce = tf.keras.losses.BinaryCrossentropy(
        from_logits=True,
        reduction=tf.keras.losses.Reduction.NONE,
    )

    @tf.function
    def fgsm_step(x_batch_tensor, y_batch_tensor):
        with tf.GradientTape() as tape:
            tape.watch(x_batch_tensor)
            logits = logit_model(x_batch_tensor, training=False)
            loss = bce(y_batch_tensor, logits)

        gradient = tape.gradient(loss, x_batch_tensor)
        masked_gradient = gradient * mask_tensor

        return tf.clip_by_value(
            x_batch_tensor + epsilon * tf.sign(masked_gradient),
            0.0,
            1.0,
        )

    for start in range(0, len(x), batch_size):
        end = start + batch_size
        x_tensor = tf.convert_to_tensor(x[start:end].astype("float32"), dtype=tf.float32)
        y_tensor = tf.convert_to_tensor(y_array[start:end].reshape(-1, 1), dtype=tf.float32)
        x_adv_list.append(fgsm_step(x_tensor, y_tensor).numpy())
        gc.collect()

    return np.concatenate(x_adv_list, axis=0).astype("float32")

def pgd_attack_batched(logit_model, x, y, epsilon, alpha, steps, feature_mask_2d, batch_size=256):
    """
    Generates masked PGD adversarial examples.

    The mask is applied at every PGD step so that blocked features remain fixed.
    """
    x_adv_list = []
    y_array = np.asarray(y).reshape(-1).astype("float32")
    mask_tensor = tf.convert_to_tensor(feature_mask_2d, dtype=tf.float32)
    bce = tf.keras.losses.BinaryCrossentropy(
        from_logits=True,
        reduction=tf.keras.losses.Reduction.NONE,
    )

    @tf.function
    def pgd_step(x_adv_tensor, x_orig_tensor, y_tensor):
        with tf.GradientTape() as tape:
            tape.watch(x_adv_tensor)
            logits = logit_model(x_adv_tensor, training=False)
            loss = bce(y_tensor, logits)

        gradient = tape.gradient(loss, x_adv_tensor)
        masked_gradient = gradient * mask_tensor

        x_adv_tensor = x_adv_tensor + alpha * tf.sign(masked_gradient)

        perturbation = tf.clip_by_value(
            x_adv_tensor - x_orig_tensor,
            -epsilon,
            epsilon,
        )

        # The mask is also applied to the perturbation for numerical safety.
        perturbation = perturbation * mask_tensor

        return tf.clip_by_value(x_orig_tensor + perturbation, 0.0, 1.0)

    for start in range(0, len(x), batch_size):
        end = start + batch_size
        x_original = tf.convert_to_tensor(x[start:end].astype("float32"), dtype=tf.float32)
        x_adv = tf.identity(x_original)
        y_tensor = tf.convert_to_tensor(y_array[start:end].reshape(-1, 1), dtype=tf.float32)

        for _ in range(steps):
            x_adv = pgd_step(x_adv, x_original, y_tensor)

        x_adv_list.append(x_adv.numpy())
        gc.collect()

    return np.concatenate(x_adv_list, axis=0).astype("float32")

def evaluate_adversarial(model, X_test_2d, y_test, epsilons, y_clean_proba, best_threshold, feature_mask_2d):
    results = []

    logit_model = tf.keras.Model(inputs=model.input, outputs=model.get_layer("logits").output)

    total_fgsm_time = 0.0
    total_pgd_time = 0.0

    for eps in epsilons:
        # FGSM - Attack and inference
        t0_fgsm = time.perf_counter()
        X_fgsm = fgsm_attack_batched(logit_model, X_test_2d, y_test, eps, feature_mask_2d, ADV_BATCH_SIZE)
        y_fgsm_proba = model.predict(X_fgsm, batch_size=ADV_BATCH_SIZE, verbose=0)
        fgsm_time = time.perf_counter() - t0_fgsm
        total_fgsm_time += fgsm_time

        # PGD - Attack and inference
        t0_pgd = time.perf_counter()
        X_pgd = pgd_attack_batched(logit_model, X_test_2d, y_test, eps, eps/4, 10, feature_mask_2d, ADV_BATCH_SIZE)
        y_pgd_proba = model.predict(X_pgd, batch_size=ADV_BATCH_SIZE, verbose=0)
        pgd_time = time.perf_counter() - t0_pgd
        total_pgd_time += pgd_time

        # FGSM metrics
        fgsm_metrics = get_metrics(y_test, y_fgsm_proba, best_threshold)
        fgsm_metrics["ASR_I"] = calculate_ASR_I(y_test, y_clean_proba, y_fgsm_proba, best_threshold)
        fgsm_metrics["time"] = fgsm_time

        # PGD metrics
        pgd_metrics = get_metrics(y_test, y_pgd_proba, best_threshold)
        pgd_metrics["ASR_I"] = calculate_ASR_I(y_test, y_clean_proba, y_pgd_proba, best_threshold)
        pgd_metrics["time"] = pgd_time

        results.append({"epsilon": eps, "FGSM": fgsm_metrics, "PGD": pgd_metrics})

        del X_fgsm, X_pgd, y_fgsm_proba, y_pgd_proba
        gc.collect()

    return results, total_fgsm_time, total_pgd_time

def run_single_experiment(X, y, feature_mask_1d, run_id, seed):
    tf.keras.backend.clear_session()
    gc.collect()
    set_seed(seed)

    X_train_2d, X_val_2d, X_test_2d, y_train_res, y_val, y_test, size, scaler = prepare_splits(X, y, seed)

    # Reshape the 1D feature mask to the same padded 2D layout used by Conv2D.
    feature_mask_2d = reshape_to_2d(np.array([feature_mask_1d], dtype="float32"), size)[0:1]

    train_ds = make_tf_dataset(X_train_2d, y_train_res, BATCH_SIZE, shuffle=True, seed=seed)
    val_ds = make_tf_dataset(X_val_2d, y_val, BATCH_SIZE, shuffle=False, seed=seed)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
    ]

    model = build_mtl_model_M1(input_shape=(size, size, 1))

    start_train = time.perf_counter()
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)
    train_time = time.perf_counter() - start_train
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

    # ------------------------------------------------------------
    # DYNAMIC THRESHOLD OPTIMIZATION (ON THE VALIDATION SET)
    # ------------------------------------------------------------
    y_val_proba = model.predict(X_val_2d.astype("float32"), batch_size=ADV_BATCH_SIZE, verbose=0)
    best_threshold = get_best_threshold(y_val, y_val_proba)

    # ------------------------------------------------------------
    # CLEAN PREDICTION AND DIAGNOSTIC
    # ------------------------------------------------------------
    y_clean_proba = model.predict(X_test_2d.astype("float32"), batch_size=ADV_BATCH_SIZE, verbose=0)
    clean_metrics = get_metrics(y_test, y_clean_proba, best_threshold)

    y_clean_pred = (y_clean_proba.reshape(-1) > best_threshold).astype("int32")
    y_test_flat = np.asarray(y_test).reshape(-1).astype("int32")
    originally_malicious = (y_test_flat == 1) & (y_clean_pred == 1)

    print("\n" + "-"*60)
    print(f"[Run Diagnostic {run_id}]")
    print(f"Best Threshold (Validation): {best_threshold:.4f}")
    print("Clean metrics:", clean_metrics)
    print("Clean probabilities - min/max/mean:", float(y_clean_proba.min()), float(y_clean_proba.max()), float(y_clean_proba.mean()))
    print("y_test distribution:", np.bincount(y_test_flat))
    print("Clean prediction distribution:", np.bincount(y_clean_pred))
    print("Correctly detected malicious samples:", originally_malicious.sum())
    print("-"*60)

    # ------------------------------------------------------------
    # ADVERSARIAL ATTACK
    # ------------------------------------------------------------
    adversarial_metrics, total_fgsm, total_pgd = evaluate_adversarial(model, X_test_2d, y_test, EPSILONS, y_clean_proba, best_threshold, feature_mask_2d)

    print(f"\n[Run {run_id} Completed] Target Time = {train_time:.2f}s")
    print(f"Accumulated time -> FGSM: {total_fgsm:.2f}s | PGD: {total_pgd:.2f}s")
    for adv in adversarial_metrics:
        print(f"   -> Eps={adv['epsilon']} | FGSM (F1={adv['FGSM']['F1']:.4f}, ASR_I={adv['FGSM']['ASR_I']:.4f}, Time={adv['FGSM']['time']:.2f}s) | PGD (F1={adv['PGD']['F1']:.4f}, ASR_I={adv['PGD']['ASR_I']:.4f}, Time={adv['PGD']['time']:.2f}s)")

    result = {
        "run": run_id, "seed": seed, "best_epoch": best_epoch, "train_time": train_time,
        "best_threshold": best_threshold, "adversarial": adversarial_metrics,
        "total_fgsm_time": total_fgsm, "total_pgd_time": total_pgd
    }

    del model, train_ds, val_ds, X_train_2d, X_val_2d, X_test_2d, y_train_res, y_val, y_test, y_clean_proba
    gc.collect()
    tf.keras.backend.clear_session()
    return result

# THE FUNCTION THAT WILL BE CALLED BY THE CHILD PROCESS
def _worker_run(run_id, seed, X, y, feature_mask_1d, return_dict):
    configure_gpu()
    result = run_single_experiment(X, y, feature_mask_1d, run_id, seed)
    return_dict[run_id] = result

# ------------------------------------------------------------
# RESULTS PROCESSING
# ------------------------------------------------------------
def summarize_adversarial_results(all_results):
    print("\n" + "=" * 80)
    print("ADVERSARIAL RESULTS (WHITE-BOX): MEAN ± STANDARD DEVIATION")
    print("=" * 80)

    thresholds = [r.get("best_threshold", 0.5) for r in all_results]
    print(f"\n[Info] Mean Optimal Threshold: {np.mean(thresholds):.4f} ± {np.std(thresholds):.4f}")

    # Display the total accumulated time
    total_fgsm = [r["total_fgsm_time"] for r in all_results]
    total_pgd = [r["total_pgd_time"] for r in all_results]

    print("\n[Total Accumulated Times (All Epsilons)]")
    print(f"  FGSM Total Time: {np.mean(total_fgsm):.4f}s ± {np.std(total_fgsm):.4f}s")
    print(f"  PGD Total Time:  {np.mean(total_pgd):.4f}s ± {np.std(total_pgd):.4f}s")
    print("-" * 80)

    for attack in ["FGSM", "PGD"]:
        print(f"\nAttack: {attack}")
        for eps in EPSILONS:
            print(f"Epsilon = {eps}")
            for metric in ["Acc", "Prec", "Rec", "F1", "AUC", "ASR_I", "time"]:
                values = [adv_result[attack][metric] for r in all_results for adv_result in r["adversarial"] if adv_result["epsilon"] == eps]
                if metric == "time":
                    print(f"  Time (Attack+Inf): {np.mean(values):.4f}s ± {np.std(values):.4f}s")
                else:
                    print(f"  {metric}: {np.mean(values):.4f} ± {np.std(values):.4f}")

def export_results_to_csv(all_results, output_path="adversarial_metrics_whitebox_only_HIKARI.csv"):
    rows = []
    for result in all_results:
        run = result["run"]
        seed = result["seed"]
        best_t = result.get("best_threshold", 0.5)
        t_fgsm_total = result.get("total_fgsm_time", 0.0)
        t_pgd_total = result.get("total_pgd_time", 0.0)

        for adv in result["adversarial"]:
            eps = adv["epsilon"]
            for attack in ["FGSM", "PGD"]:
                row = {
                    "run": run, "seed": seed, "condition": "adversarial",
                    "attack": attack, "epsilon": eps, "threshold": best_t,
                    **adv[attack],  # Includes the metrics + the time key for this epsilon
                    "best_epoch": result["best_epoch"],
                    "train_time": result["train_time"],
                    "total_accumulated_time": t_fgsm_total if attack == "FGSM" else t_pgd_total
                }
                rows.append(row)
    df_results = pd.DataFrame(rows)
    df_results.to_csv(output_path, index=False)
    print(f"\nResults saved to: {output_path}")
    return df_results

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main(df_filtrado, feature_mask_1d=None):
    """
    Executes N_RUNS runs of white-box adversarial evaluation with a HIKARI
    plausibility mask.

    Accepted dataframe formats:
    1. Full HIKARI dataframe:
       metadata columns + 79 features + traffic_category + Label.
    2. Filtered HIKARI dataframe:
       79 features + traffic_category + Label.

    traffic_category and Label are response variables and are never used as
    input features or as part of the plausibility mask.
    """
    try:
        mp.set_start_method('spawn')
    except RuntimeError:
        pass

    df = df_filtrado.copy()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)

    X, y, feature_cols = select_hikari_features_and_target(df)

    if feature_mask_1d is None:
        feature_mask_1d, mask_series = build_hikari_plausibility_mask(feature_cols)
        print_plausibility_mask_report(mask_series)
    else:
        if isinstance(feature_mask_1d, pd.Series):
            # Align a named external mask with the inferred HIKARI feature order.
            feature_mask_1d = feature_mask_1d.reindex(feature_cols).values

        feature_mask_1d = np.asarray(feature_mask_1d, dtype="float32").reshape(-1)

        if len(feature_mask_1d) != X.shape[1]:
            raise ValueError(
                f"feature_mask_1d has length {len(feature_mask_1d)}, "
                f"but X has {X.shape[1]} features."
            )

        if np.isnan(feature_mask_1d).any():
            raise ValueError(
                "feature_mask_1d contains NaN values. If a pandas Series was "
                "provided, make sure its index matches the HIKARI feature names."
            )

        invalid_values = set(np.unique(feature_mask_1d)) - {0.0, 1.0}
        if invalid_values:
            raise ValueError(
                "feature_mask_1d must be binary, with values 0.0 or 1.0. "
                f"Found invalid values: {sorted(invalid_values)}"
            )

        mask_series = pd.Series(feature_mask_1d, index=feature_cols, dtype="float32")
        print_plausibility_mask_report(mask_series)

    all_results = []
    manager = mp.Manager()
    return_dict = manager.dict()

    print("\n" + "="*50)
    print(f"STARTING {N_RUNS} RUNS (WHITE-BOX WITH DYNAMIC THRESHOLD)")
    print("="*50)

    for run in range(N_RUNS):
        seed = 42 + run
        print(f"\n[{time.strftime('%H:%M:%S')}] Isolated Process -> Run {run + 1}/{N_RUNS}")

        p = mp.Process(target=_worker_run, args=(run + 1, seed, X, y, feature_mask_1d, return_dict))
        p.start()
        p.join()

        if (run + 1) in return_dict:
            all_results.append(return_dict[run + 1])
            del return_dict[run + 1]

    summarize_adversarial_results(all_results)
    df_results = export_results_to_csv(all_results)
    return all_results, df_results


In [ ]:
import importlib
import ids_engine2

importlib.reload(ids_engine2)

In [ ]:
all_results, df_results = ids_engine2.main(df_filtrado)